In [94]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import fmin
import math

import yfinance as yf
import pandas as pd
import pandas_market_calendars as mcal

## Black Scholes Pricing and Implied Volatility

In [196]:
def black_scholes(S, K, t, r, sigma, put_or_call='call'):
    d1 = (math.log(S/K) + (r + sigma**2 / 2) * t)/(sigma * np.sqrt(t))
    d2 = (math.log(S/K) + (r - sigma**2 / 2) * t)/(sigma * np.sqrt(t))
    if put_or_call == 'call':
        return S * norm.cdf(d1) - K * math.e**(-r*t) * norm.cdf(d2)
    elif put_or_call == 'put':
        return K * math.e**(-r*t) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        return None

def black_scholes_df(row, S, r, put_or_call='call'):
    K = row['strike']
    t = row['remainingTradeDate']/365
    sigma = row['calculated_iv']

    return black_scholes(S, K, t, r, sigma, put_or_call)

def implied_volatility(s, option_price, S, K, t, r, put_or_call ='call' ):    
    d1 = (math.log(S/K) + (r + s[0]**2 / 2) * t)/(s[0] * np.sqrt(t))
    d2 = (math.log(S/K) + (r - s[0]**2 / 2) * t)/(s[0] * np.sqrt(t))
    calculated_price = black_scholes(S, K, t, r, s[0], put_or_call)

    print('s : ', s)
    print('calculated_price')
    if put_or_call == 'call':
        of = (S * norm.cdf(d1) - K * math.e**(-r*t) * norm.cdf(d2)) - option_price
        val = of**2    
    elif put_or_call == 'put':
        of = (K * math.e**(-r*t) * norm.cdf(-d2) - S * norm.cdf(-d1)) - option_price
        val = of**2
    print(val)

    return val

def iv_objective(s, option_price, S, K, t, r, put_or_call):
    sigma = abs(s[0])
    calculated_price = black_scholes(S, K, t, r, sigma, put_or_call)
    return (calculated_price - option_price)**2

def implied_volatility_df(row, S, r, put_or_call='call', s0=5):
    # print('row : ', row)
    K = row['strike']
    option_price = row['lastPrice']
    t = row['remainingTradeDate']/365

    result = fmin(iv_objective, [s0], args=(option_price, S, K, t, r, put_or_call), disp=False)

    return abs(result[0])
    

In [69]:
S = 100
K = 105
t = 0.5
r = 0.05
sigma = 0.2

black_scholes(S, K, t, r, sigma, 'put')

np.float64(6.989220930514911)

In [70]:
# implied_volatility([0.3], price, S, K, t, r, 'call')

In [158]:
# bs_price = 6.989220930514911
# iv = fmin(implied_volatility, [0.3], args=(bs_price, S, K, t, r, 'call'))

In [159]:
print(iv)

[2.05426025]


## Gaterhing Options Data

In [65]:
ticker = yf.Ticker('AAPL')

In [50]:
ticker.fast_info['last_price']

332.2699890136719

In [51]:
df_call = None
df_put = None
for exp in ticker.options:
    opt = ticker.option_chain(exp)
    call = opt.calls
    call['exp'] = exp
    df_call = pd.concat([df_call, call], ignore_index=True)
    put = opt.puts
    put['exp'] = exp
    df_put = pd.concat([df_put, put], ignore_index=True)

In [79]:
black_scholes(S, K, t, r, 1.886719, 'call')

np.float64(87.5317238616764)

In [153]:
S = 332.2699890136719
K = 245
t = 2/365
r = 0.0402
bs_price = 87.66

iv = fmin(implied_volatility, [5], args=(bs_price, S, K, t, r, 'call'))
print('iv : ', iv)

s :  [5.]
calculated_price
136.6299827988437
s :  [5.25]
calculated_price
173.89014688880386
s :  [4.75]
calculated_price
104.82864289622826
s :  [4.5]
calculated_price
78.18777097442052
s :  [4.]
calculated_price
38.976719367079454
s :  [3.5]
calculated_price
15.710140800733708
s :  [2.5]
calculated_price
0.5338845745300392
s :  [1.5]
calculated_price
0.11172091928151029
s :  [-0.5]
calculated_price
7684.275599999999
s :  [2.5]
calculated_price
0.5338845745300392
s :  [0.5]
calculated_price
0.13188746799876996
s :  [1.]
calculated_price
0.1318228521583977
s :  [2.]
calculated_price
0.0033537737566396307
s :  [2.5]
calculated_price
0.5338845745300392
s :  [2.5]
calculated_price
0.5338845745300392
s :  [1.75]
calculated_price
0.06130729971585899
s :  [2.25]
calculated_price
0.06962739405817785
s :  [1.875]
calculated_price
0.02817214901269847
s :  [2.125]
calculated_price
0.007277617405551251
s :  [2.0625]
calculated_price
8.809018075634333e-05
s :  [2.125]
calculated_price
0.0072776174

In [188]:
cal = mcal.get_calendar('NYSE')
def trading_days(start_date, end_date, cal):
    
    schedule = cal.schedule(start_date, end_date)
    trading_days = mcal.date_range(schedule, frequency='1D').date

    return len(trading_days)

def trading_days_df(row, cal):
    # print('tes')
    # print('row : ', row)
    start_date = row['lastTradeDate']
    # print('start_date : ', start_date)
    end_date = row['exp']
    # print('end_date : ', end_date)
    schedule = cal.schedule(start_date, end_date)
    trading_days = mcal.date_range(schedule, frequency='1D').date

    return len(trading_days)

def trading_days_df_future(row, cal):
    # print('tes')
    # print('row : ', row)
    start_date = row['futureDate']
    # print('start_date : ', start_date)
    end_date = row['exp']
    # print('end_date : ', end_date)
    schedule = cal.schedule(start_date, end_date)
    trading_days = mcal.date_range(schedule, frequency='1D').date

    return len(trading_days)

In [155]:
df_call['lastTradeDate'][0:10]

0   2026-09-11 19:46:47+00:00
1   2026-09-11 15:38:06+00:00
2   2026-09-11 19:54:10+00:00
3   2026-09-11 19:38:49+00:00
4   2026-09-11 19:50:57+00:00
5   2026-09-11 19:58:19+00:00
6   2026-09-11 19:49:10+00:00
7   2026-09-11 19:59:52+00:00
8   2026-09-11 19:59:46+00:00
9   2026-09-11 19:59:43+00:00
Name: lastTradeDate, dtype: datetime64[s, UTC]

In [156]:
trading_days('2026-09-11', '2026-09-14', cal)

2

In [147]:
df_call.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,exp,remainingTradeDate
0,AAPL260914C00245000,2026-09-11 19:46:47+00:00,245.0,87.66,85.75,89.2,9.510002,12.168908,20.0,1,1.886719,True,REGULAR,USD,2026-09-14,2
1,AAPL260914C00285000,2026-09-11 15:38:06+00:00,285.0,50.20,45.80,48.4,4.930000,10.890215,1.0,12,1.449222,True,REGULAR,USD,2026-09-14,2
2,AAPL260914C00290000,2026-09-11 19:54:10+00:00,290.0,42.60,41.50,42.6,12.099998,39.672127,23.0,10,1.018560,True,REGULAR,USD,2026-09-14,2
3,AAPL260914C00295000,2026-09-11 19:38:49+00:00,295.0,37.52,36.10,38.6,7.520001,25.066668,8.0,35,0.726565,True,REGULAR,USD,2026-09-14,2
4,AAPL260914C00300000,2026-09-11 19:50:57+00:00,300.0,32.45,31.15,33.4,7.270001,28.872124,309.0,40,1.062016,True,REGULAR,USD,2026-09-14,2


In [138]:
df_call['remainingTradeDate'] = df_call.apply(trading_days_df, axis=1, cal=cal)

In [142]:
df_put['remainingTradeDate'] = df_put.apply(trading_days_df, axis=1, cal=cal)

In [165]:
df_call.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,exp,remainingTradeDate
0,AAPL260914C00245000,2026-09-11 19:46:47+00:00,245.0,87.66,85.75,89.2,9.510002,12.168908,20.0,1,1.886719,True,REGULAR,USD,2026-09-14,2
1,AAPL260914C00285000,2026-09-11 15:38:06+00:00,285.0,50.20,45.80,48.4,4.930000,10.890215,1.0,12,1.449222,True,REGULAR,USD,2026-09-14,2
2,AAPL260914C00290000,2026-09-11 19:54:10+00:00,290.0,42.60,41.50,42.6,12.099998,39.672127,23.0,10,1.018560,True,REGULAR,USD,2026-09-14,2
3,AAPL260914C00295000,2026-09-11 19:38:49+00:00,295.0,37.52,36.10,38.6,7.520001,25.066668,8.0,35,0.726565,True,REGULAR,USD,2026-09-14,2
4,AAPL260914C00300000,2026-09-11 19:50:57+00:00,300.0,32.45,31.15,33.4,7.270001,28.872124,309.0,40,1.062016,True,REGULAR,USD,2026-09-14,2


In [168]:
df_call['strike']

0       245.0
1       285.0
2       290.0
3       295.0
4       300.0
        ...  
1303    560.0
1304    570.0
1305    580.0
1306    590.0
1307    600.0
Name: strike, Length: 1308, dtype: float64

In [182]:
df_call['calculated_iv'] = df_call.apply(implied_volatility_df, axis=1, S=S, r=r, put_or_call='call')
df_put['calculated_iv'] = df_put.apply(implied_volatility_df, axis=1, S=S, r=r, put_or_call='put')


/var/folders/fq/4g0b7nkd5v1d0krt8k9z5_ww0000gn/T/ipykernel_915/795459077.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  d1 = (math.log(S/K) + (r + sigma**2 / 2) * t)/(sigma * np.sqrt(t))
/var/folders/fq/4g0b7nkd5v1d0krt8k9z5_ww0000gn/T/ipykernel_915/795459077.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  d2 = (math.log(S/K) + (r - sigma**2 / 2) * t)/(sigma * np.sqrt(t))
/var/folders/fq/4g0b7nkd5v1d0krt8k9z5_ww0000gn/T/ipykernel_915/795459077.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  d1 = (math.log(S/K) + (r + sigma**2 / 2) * t)/(sigma * np.sqrt(t))
/var/folders/fq/4g0b7nkd5v1d0krt8k9z5_ww0000gn/T/ipykernel_915/795459077.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  d2 = (math.log(S/K) + (r - sigma**2 / 2) * t)/(sigma * np.sqrt(t))


In [187]:
df_call[10:20]

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,exp,remainingTradeDate,calculated_iv
10,AAPL260914C00330000,2026-09-11 19:59:58+00:00,330.0,3.40,3.30,3.55,1.49,78.010480,18399.0,5843,0.228523,True,REGULAR,USD,2026-09-14,2,0.208984
11,AAPL260914C00335000,2026-09-11 19:59:59+00:00,335.0,1.02,0.98,1.05,0.25,32.467533,61462.0,2911,0.217537,False,REGULAR,USD,2026-09-14,2,0.212891
12,AAPL260914C00340000,2026-09-11 19:59:59+00:00,340.0,0.20,0.20,0.24,-0.10,-33.333336,26076.0,2511,0.231453,False,REGULAR,USD,2026-09-14,2,0.220459
13,AAPL260914C00345000,2026-09-11 19:59:58+00:00,345.0,0.06,0.05,0.06,-0.06,-50.000000,9379.0,714,0.257820,False,REGULAR,USD,2026-09-14,2,0.257568
14,AAPL260914C00350000,2026-09-11 19:59:58+00:00,350.0,0.02,0.02,0.03,-0.02,-50.000000,5375.0,1760,0.308601,False,REGULAR,USD,2026-09-14,2,0.291931
15,AAPL260914C00355000,2026-09-11 19:56:17+00:00,355.0,0.01,0.00,0.02,-0.01,-50.000000,961.0,272,0.359381,False,REGULAR,USD,2026-09-14,2,0.334473
16,AAPL260914C00360000,2026-09-11 19:20:01+00:00,360.0,0.02,0.00,0.02,0.00,0.000000,684.0,496,0.429693,False,REGULAR,USD,2026-09-14,2,0.427490
17,AAPL260914C00365000,2026-09-11 13:43:27+00:00,365.0,0.01,0.00,0.04,-0.02,-66.666670,40.0,88,0.535161,False,REGULAR,USD,2026-09-14,2,0.458008
18,AAPL260914C00370000,2026-09-11 14:48:22+00:00,370.0,0.01,0.00,0.14,0.00,0.000000,15.0,277,0.644535,False,REGULAR,USD,2026-09-14,2,0.517212
19,AAPL260914C00375000,2026-09-11 14:41:42+00:00,375.0,0.01,0.00,0.04,0.00,0.000000,4.0,14,0.617191,False,REGULAR,USD,2026-09-14,2,0.575012


In [186]:
df_put.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,exp,remainingTradeDate,calculated_iv
0,AAPL260914P00250000,2026-09-11 13:36:13+00:00,250.0,0.01,0.0,0.06,0.00,0.0,9501.0,3,1.414065,False,REGULAR,USD,2026-09-14,2,1.276855
1,AAPL260914P00255000,2026-09-11 13:36:33+00:00,255.0,0.01,0.0,0.01,0.00,0.0,10500.0,373,1.125004,False,REGULAR,USD,2026-09-14,2,1.194397
2,AAPL260914P00260000,2026-09-11 17:20:15+00:00,260.0,0.01,0.0,0.02,0.00,0.0,1.0,369,1.109379,False,REGULAR,USD,2026-09-14,2,1.113159
3,AAPL260914P00265000,2026-09-11 19:47:43+00:00,265.0,0.01,0.0,0.01,0.00,0.0,5.0,161,0.968750,False,REGULAR,USD,2026-09-14,2,1.033203
4,AAPL260914P00270000,2026-09-11 19:47:43+00:00,270.0,0.03,0.0,0.05,0.01,50.0,2.0,33,1.039067,False,REGULAR,USD,2026-09-14,2,1.061340


In [192]:
df_call_future

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,exp,remainingTradeDate,calculated_iv,futureDate
0,AAPL261016C00130000,2026-09-09 15:11:58+00:00,130.0,185.00,200.90,204.80,0.00,0.000000,1.0,34,1.460452,True,REGULAR,USD,2026-10-16,28,0.000000,2026-10-10
1,AAPL261016C00135000,2026-09-01 14:53:41+00:00,135.0,190.63,196.35,199.85,0.00,0.000000,1.0,61,1.485842,True,REGULAR,USD,2026-10-16,33,0.000000,2026-10-10
2,AAPL261016C00140000,2026-09-01 18:37:22+00:00,140.0,186.05,191.30,194.85,0.00,0.000000,2.0,24,1.423831,True,REGULAR,USD,2026-10-16,33,0.000000,2026-10-10
3,AAPL261016C00145000,2026-08-04 14:21:00+00:00,145.0,162.99,174.30,177.65,0.00,0.000000,9.0,18,0.000010,True,REGULAR,USD,2026-10-16,53,0.000000,2026-10-10
4,AAPL261016C00150000,2026-09-01 15:13:09+00:00,150.0,173.50,181.30,184.90,0.00,0.000000,1.0,18,1.326175,True,REGULAR,USD,2026-10-16,33,0.000000,2026-10-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1030,AAPL281215C00560000,2026-09-10 15:03:29+00:00,560.0,11.91,12.75,14.00,0.00,0.000000,1.0,94,0.317084,False,REGULAR,USD,2028-12-15,572,0.348450,2026-10-10
1031,AAPL281215C00570000,2026-09-11 14:45:44+00:00,570.0,13.35,11.95,13.00,3.05,29.611652,1.0,48,0.316199,False,REGULAR,USD,2028-12-15,571,0.369507,2026-10-10
1032,AAPL281215C00580000,2026-09-11 19:46:52+00:00,580.0,11.97,11.15,12.35,1.67,16.213593,10.0,153,0.317527,False,REGULAR,USD,2028-12-15,571,0.364685,2026-10-10
1033,AAPL281215C00590000,2026-09-11 13:45:32+00:00,590.0,11.09,10.15,11.70,1.53,16.004180,11.0,205,0.318504,False,REGULAR,USD,2028-12-15,571,0.363770,2026-10-10


In [205]:
S

332.2699890136719

## Future Option Price

In [198]:
future_S = 400
r = 0.0402

future_date = '2026-10-10'

df_call_future = df_call[df_call['exp'] > future_date].reset_index(drop=True)
df_put_future = df_put[df_put['exp'] > future_date].reset_index(drop=True)

df_call_future['futureDate'] = future_date
df_put_future['futureDate'] = future_date

df_call_future['remainingTradeDate'] = df_call_future.apply(trading_days_df_future, axis=1, cal=cal)
df_put_future['remainingTradeDate'] = df_put_future.apply(trading_days_df_future, axis=1, cal=cal)


In [200]:
df_call_future['future_option_price'] = df_call_future.apply(black_scholes_df, axis=1, S=future_S, r=r, put_or_call='call')

/var/folders/fq/4g0b7nkd5v1d0krt8k9z5_ww0000gn/T/ipykernel_915/2013982487.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  d1 = (math.log(S/K) + (r + sigma**2 / 2) * t)/(sigma * np.sqrt(t))
/var/folders/fq/4g0b7nkd5v1d0krt8k9z5_ww0000gn/T/ipykernel_915/2013982487.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  d2 = (math.log(S/K) + (r - sigma**2 / 2) * t)/(sigma * np.sqrt(t))


In [202]:
df_put_future['future_option_price'] = df_put_future.apply(black_scholes_df, axis=1, S=future_S, r=r, put_or_call='put')





/var/folders/fq/4g0b7nkd5v1d0krt8k9z5_ww0000gn/T/ipykernel_915/2013982487.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  d1 = (math.log(S/K) + (r + sigma**2 / 2) * t)/(sigma * np.sqrt(t))
/var/folders/fq/4g0b7nkd5v1d0krt8k9z5_ww0000gn/T/ipykernel_915/2013982487.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  d2 = (math.log(S/K) + (r - sigma**2 / 2) * t)/(sigma * np.sqrt(t))


In [203]:
df_call_future['price_return'] = ((df_call_future['future_option_price'] /  df_call_future['lastPrice']) - 1) * 100
df_put_future['price_return'] = ((df_put_future['future_option_price'] /  df_put_future['lastPrice']) - 1) * 100

In [207]:
df_call_future.head(50)

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,exp,remainingTradeDate,calculated_iv,futureDate,future_option_price,price_return
0,AAPL261016C00130000,2026-09-09 15:11:58+00:00,130.0,185.00,200.90,204.80,0.000000,0.000000,1.0,34,1.460452,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,270.071569,45.984632
1,AAPL261016C00135000,2026-09-01 14:53:41+00:00,135.0,190.63,196.35,199.85,0.000000,0.000000,1.0,61,1.485842,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,265.074322,39.051735
2,AAPL261016C00140000,2026-09-01 18:37:22+00:00,140.0,186.05,191.30,194.85,0.000000,0.000000,2.0,24,1.423831,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,260.077075,39.788807
3,AAPL261016C00145000,2026-08-04 14:21:00+00:00,145.0,162.99,174.30,177.65,0.000000,0.000000,9.0,18,0.000010,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,255.079827,56.500293
4,AAPL261016C00150000,2026-09-01 15:13:09+00:00,150.0,173.50,181.30,184.90,0.000000,0.000000,1.0,18,1.326175,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,250.082580,44.139816
5,AAPL261016C00155000,2026-08-28 17:36:20+00:00,155.0,165.77,176.40,179.85,0.000000,0.000000,2.0,62,1.282718,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,245.085333,47.846614
6,AAPL261016C00160000,2026-09-09 15:00:54+00:00,160.0,154.20,171.45,174.85,0.000000,0.000000,1.0,232,1.240238,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,240.088085,55.699147
7,AAPL261016C00165000,2026-09-11 15:52:43+00:00,165.0,170.40,166.40,169.90,18.699997,12.326960,1.0,81,1.193363,True,REGULAR,USD,2026-10-16,5,1.719788,2026-10-10,235.090895,37.964141
8,AAPL261016C00170000,2026-09-08 19:55:28+00:00,170.0,147.03,161.45,164.90,0.000000,0.000000,3.0,63,1.153080,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,230.093591,56.494315
9,AAPL261016C00175000,2026-08-07 14:29:26+00:00,175.0,139.72,156.45,159.95,0.000000,0.000000,2.0,25,1.113774,True,REGULAR,USD,2026-10-16,5,0.000000,2026-10-10,225.096343,61.105313


In [208]:
df_put_future.head(50)

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,exp,remainingTradeDate,calculated_iv,futureDate,future_option_price,price_return
0,AAPL261016P00130000,2026-08-11 13:49:18+00:00,130.0,0.02,0.00,0.52,0.000000,0.000000,46.0,66,1.309574,False,REGULAR,USD,2026-10-16,5,0.849854,2026-10-10,1.223359e-29,-100.000000
1,AAPL261016P00135000,2026-07-16 17:24:05+00:00,135.0,0.05,0.00,1.01,0.000000,0.000000,2.0,70,1.377933,False,REGULAR,USD,2026-10-16,5,0.757507,2026-10-10,1.269589e-34,-100.000000
2,AAPL261016P00140000,2026-09-09 15:36:41+00:00,140.0,0.01,0.00,0.06,0.000000,0.000000,10.0,618,0.964844,False,REGULAR,USD,2026-10-16,5,0.971680,2026-10-10,3.640207e-20,-100.000000
3,AAPL261016P00145000,2026-09-08 16:53:46+00:00,145.0,0.03,0.00,0.04,0.000000,0.000000,3.0,395,0.898439,False,REGULAR,USD,2026-10-16,5,1.005615,2026-10-10,1.014384e-17,-100.000000
4,AAPL261016P00150000,2026-09-04 15:43:40+00:00,150.0,0.02,0.00,0.05,0.000000,0.000000,30.0,400,0.875001,False,REGULAR,USD,2026-10-16,5,0.917419,2026-10-10,8.794527e-20,-100.000000
5,AAPL261016P00155000,2026-09-09 15:22:45+00:00,155.0,0.01,0.00,0.01,0.000000,0.000000,28.0,170,0.750003,False,REGULAR,USD,2026-10-16,5,0.862305,2026-10-10,7.198527e-21,-100.000000
6,AAPL261016P00160000,2026-09-11 14:54:36+00:00,160.0,0.01,0.00,0.01,0.000000,0.000000,2.0,260,0.718753,False,REGULAR,USD,2026-10-16,5,0.859253,2026-10-10,1.055115e-19,-100.000000
7,AAPL261016P00165000,2026-09-08 16:43:49+00:00,165.0,0.15,0.00,0.52,0.000000,0.000000,1.0,230,0.998047,False,REGULAR,USD,2026-10-16,5,1.011780,2026-10-10,1.431499e-13,-100.000000
8,AAPL261016P00170000,2026-09-03 13:47:12+00:00,170.0,0.09,0.00,0.03,0.000000,0.000000,2.0,308,0.710940,False,REGULAR,USD,2026-10-16,5,0.887512,2026-10-10,2.685602e-16,-100.000000
9,AAPL261016P00175000,2026-09-08 14:35:20+00:00,175.0,0.02,0.00,0.08,0.000000,0.000000,2.0,623,0.746096,False,REGULAR,USD,2026-10-16,5,0.761353,2026-10-10,2.038413e-20,-100.000000
